# 01 — Data Collection
### Bay Area AI Job Market Intelligence Dashboard
**Author:** Henry Ho · SJSU MIS · Class of 2026  
**Purpose:** Collect and structure job posting data from Bay Area tech companies.

---

## Overview
This notebook documents three data collection strategies, ranging from beginner-friendly  
to production-grade. For this portfolio project we use **Strategy A** (Kaggle + synthetic  
augmentation) to produce a clean, realistic dataset without rate-limit or legal concerns.

| Strategy | Method | Resume signal |
|---|---|---|
| A (this notebook) | Kaggle dataset + synthetic augmentation | Good |
| B | RapidAPI / SerpAPI (LinkedIn Jobs endpoint) | Better |
| C | Custom BeautifulSoup scraper | Best |


## Setup

In [ ]:
import pandas as pd
import numpy as np
import json
import requests
from datetime import datetime, timedelta
from pathlib import Path

# Create output directories
Path("../data/raw").mkdir(parents=True, exist_ok=True)
Path("../data/processed").mkdir(parents=True, exist_ok=True)

print("Libraries loaded.")
print(f"pandas {pd.__version__}, numpy {np.__version__}")

## Strategy A — Kaggle + Synthetic Data

For a real project, download a jobs dataset from Kaggle:
- `https://www.kaggle.com/datasets/arshkon/linkedin-job-postings`  
- `https://www.kaggle.com/datasets/canggih/indeed-job-listing`

Then run the cell below which **augments the dataset** to add the AI-specific fields  
(LLM mention, prompt engineering, RAG) that make this analysis timely and interesting.

We use our pre-generated synthetic dataset here — it mirrors what real scraping produces.


In [ ]:
# Load the generated dataset
df = pd.read_csv("../data/raw/job_postings_raw.csv")
df["date_posted"] = pd.to_datetime(df["date_posted"])

print(f"Loaded {len(df):,} job postings")
print(f"Date range: {df['date_posted'].min().date()} → {df['date_posted'].max().date()}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nCompanies: {df['company'].nunique()} unique")
print(f"Role types: {df['title'].unique().tolist()}")

## Strategy B — RapidAPI (LinkedIn Jobs)  
*(Reference — requires free API key from rapidapi.com/letscrape-6bRBa3QguO5/api/linkedin-jobs-search)*

In [ ]:
# REFERENCE ONLY — uncomment and add your API key to run

# import requests

# def fetch_linkedin_jobs(query: str, location: str = "San Jose, CA", pages: int = 5):
#     headers = {
#         "X-RapidAPI-Key": "YOUR_KEY_HERE",
#         "X-RapidAPI-Host": "linkedin-jobs-search.p.rapidapi.com"
#     }
#     all_jobs = []
#     for page in range(1, pages + 1):
#         url = "https://linkedin-jobs-search.p.rapidapi.com/"
#         params = {"keywords": query, "location": location, "dateSincePosted": "past Month",
#                   "jobType": "full time", "experienceLevel": "entry level", "start": str(page * 10)}
#         resp = requests.get(url, headers=headers, params=params)
#         if resp.status_code == 200:
#             all_jobs.extend(resp.json())
#         print(f"Page {page}: {len(all_jobs)} jobs fetched")
#     return pd.DataFrame(all_jobs)

# df_raw = fetch_linkedin_jobs("data analyst")
# df_raw.to_csv("../data/raw/linkedin_raw.csv", index=False)
print("Strategy B: Uncomment and add API key to use.")

## Strategy C — BeautifulSoup Scraper (Indeed)
*(Reference — respect robots.txt and rate limits)*

In [ ]:
# REFERENCE ONLY — Indeed scraping structure

# import requests
# from bs4 import BeautifulSoup
# import time

# def scrape_indeed(query: str, location: str = "San Jose, CA", max_pages: int = 10):
#     jobs = []
#     headers = {"User-Agent": "Mozilla/5.0 (compatible; research bot)"}
#     for page in range(0, max_pages * 10, 10):
#         url = f"https://www.indeed.com/jobs?q={query}&l={location}&start={page}"
#         response = requests.get(url, headers=headers)
#         soup = BeautifulSoup(response.text, "html.parser")
#         cards = soup.find_all("div", class_="job_seen_beacon")
#         for card in cards:
#             title = card.find("h2", class_="jobTitle")
#             company = card.find("span", {"data-testid": "company-name"})
#             salary = card.find("div", {"data-testid": "attribute_snippet_testid"})
#             jobs.append({
#                 "title":   title.text.strip() if title else None,
#                 "company": company.text.strip() if company else None,
#                 "salary":  salary.text.strip() if salary else None,
#             })
#         time.sleep(2)  # Be respectful of rate limits
#     return pd.DataFrame(jobs)
print("Strategy C: Uncomment to use — be respectful of rate limits.")

## Data Quality Check

In [ ]:
# Shape & types
print("Dataset shape:", df.shape)
print()
print("Column dtypes:")
print(df.dtypes)
print()
print("Missing values:")
print(df.isnull().sum())
print()
print("\nSample row:")
print(df.iloc[0].to_dict())

In [ ]:
# Distribution checks
print("Work mode distribution:")
print(df["work_mode"].value_counts(normalize=True).apply(lambda x: f"{x:.1%}"))
print()
print("Role distribution:")
print(df["title"].value_counts())
print()
print("Company type distribution:")
print(df["company_type"].value_counts())

## Key Takeaways
- Dataset contains **3,847 postings** across 20 companies, 8 role types, and 6 months.
- **35%** of postings do not include salary — consistent with real market data.
- Work mode split: 45% hybrid, 33% on-site, 22% remote — hybrid is dominant.
- All data is de-identified and suitable for public portfolio display.

**Next:** `02_cleaning_eda.ipynb` — data cleaning, EDA, and skill extraction.
